# Домашнее задание 9. Проектирование ML-системы для расчета складских запасов

Generated at: 2026-05-24 17:22:41 MSK

**Коротко:** делаю batch + reactive retraining pipeline.

Airflow отвечает за ML-процесс. CI/CD проверяет код и Terraform. ADR фиксирует MDD-решение.


## Карта решения

- `dags/inventory_retrain_dag.py` - Airflow DAG
- `src/` - код загрузки / обучения / оценки / registry
- `infra/` - Terraform
- `reports/` - evidence
- `adr/0001-latency-mdd-decision.md` - MDD решение
- `data/` - synthetic datasets
- `.github/workflows/dz9-checks.yml` - CI/CD checks


## 1. Сравнение архитектур ML-конвейеров

| Архитектура | Когда подходит | Плюсы | Минусы | Подходит для запасов |
|---|---|---|---|---|
| Batch retraining | данные приходят пачками раз в день / час | просто / надежно / легко проверять | не мгновенная реакция | да |
| Reactive retraining | обучение стартует при появлении batch-файла или алерта | реагирует на новые данные | нужен sensor / контроль дублей | да |
| Online learning | модель обновляется почти на каждом событии | быстро адаптируется | сложнее rollback / quality gate | нет для учебного минимума |
| Streaming pipeline | данные идут постоянным потоком | near real-time | Kafka/Flink/Spark усложняют систему | только если нужны секунды |
| Human-in-the-loop approval | человек подтверждает выкладку | меньше риск плохого auto-deploy | медленнее обновление | да как safety gate |

**Вывод:**

- выбираю batch + reactive retraining через S3/File sensor
- складские выгрузки обычно приходят пачками
- задержка в минуты/часы норм
- Airflow умеет schedule / sensor / retry / branch
- CI/CD не должен быть ежедневным scheduler для обучения


## 2. Airflow DAG

Схема DAG:

```text
wait_for_inventory_batch (FileSensor locally / S3KeySensor in production)
  -> load_inventory_data
  -> validate_inventory_data
  -> train_model
  -> evaluate_model
  -> compare_with_baseline
  -> register_model / skip_deploy
```

В production вместо локального файла используется тот же task `wait_for_inventory_batch`, но с `DZ9_USE_S3_SENSOR=1`:

```text
bucket = inventory-batches
key = incoming/{{ ds }}/inventory.csv
```


In [1]:
from pathlib import Path

dag_code = Path("dags/inventory_retrain_dag.py").read_text(encoding="utf-8")
print(dag_code[:3500])


from __future__ import annotations

import os
from datetime import datetime, timedelta
from pathlib import Path

try:
    from airflow import DAG
    from airflow.operators.empty import EmptyOperator
    from airflow.operators.python import BranchPythonOperator, PythonOperator
    from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor
    from airflow.sensors.filesystem import FileSensor
except ImportError:
    class DAG:
        def __init__(self, *args, **kwargs):
            pass

        def __enter__(self):
            return self

        def __exit__(self, *args):
            return False

    class _Task:
        def __init__(self, *args, **kwargs):
            self.task_id = kwargs.get("task_id", args[0] if args else self.__class__.__name__)

        def __rshift__(self, other):
            return other

    class EmptyOperator(_Task):
        pass

    class PythonOperator(_Task):
        pass

    class BranchPythonOperator(_Task):
        pass

    class FileSensor

## Evidence Airflow

- Graph View: `reports/airflow_graph.png`
- Successful run: `reports/airflow_successful_run.png`
- Sensor log: `reports/airflow_sensor_log.md`
- Compare log: `reports/airflow_compare_log.md`
- Registry log: `reports/airflow_registry_log.md`

![Airflow graph](reports/airflow_graph.png)

**Вывод:**

- DAG содержит sensor / validation / train / evaluate / branch
- плохая модель не регистрируется автоматически
- S3 tracking показан в DAG через переключатель `DZ9_USE_S3_SENSOR=1`


## 3. Модель и metric gate

Модель простая, т.к. тут главное не качество прогноза, а MLOps-контур:

- `LinearRegression`
- target: `stock_qty_next_day`
- features: `store_id`, `sku_id`, `stock_qty`, `sales_qty`, `delivery_qty`, `day_of_week`
- baseline: `stock_qty - sales_qty + delivery_qty`
- gate: `new_mape <= 15` и `new_mape <= baseline_mape`


In [2]:
from pathlib import Path

print(Path("reports/data_validation.md").read_text(encoding="utf-8"))
print(Path("reports/model_metrics.md").read_text(encoding="utf-8"))
print(Path("reports/airflow_compare_log.md").read_text(encoding="utf-8"))


Generated at: 2026-05-24 17:22:41 MSK

# Data validation

- status: `pass`
- rows: `1920`
- stores: `4`
- sku: `8`
- errors: `[]`

**Вывод:** batch подходит под контракт: date/store/sku/stock/sales/delivery.

Generated at: 2026-05-24 17:22:41 MSK

# Model metrics

- new_mape: `1.294`
- baseline_mape: `5.409`
- new_rmse: `1.595`
- baseline_rmse: `5.786`

**Вывод:** новая модель сравнивается с baseline, без этого registry не обновляем.

Generated at: 2026-05-24 17:22:41 MSK

# Compare task log

- policy: `new_mape <= 15.0` and `new_mape <= baseline_mape`
- new_mape: `1.294`
- baseline_mape: `5.409`
- branch: `register_model`

**Вывод:** BranchPythonOperator отправляет run в register или skip.



## 4. IaC / Terraform

Минимальная infrastructure-as-code часть:

- storage для batch-файлов
- artifact storage для моделей / метрик
- Airflow config manifest
- MLflow/local registry path

В учебной сдаче это local provider. В production меняется provider, но структура остается.


In [3]:
from pathlib import Path

print(Path("infra/main.tf").read_text(encoding="utf-8"))
print("\n--- terraform plan evidence ---")
print(Path("reports/terraform_plan.txt").read_text(encoding="utf-8")[:1800])
print("\n--- terraform destroy evidence ---")
print(Path("reports/terraform_destroy_plan.txt").read_text(encoding="utf-8")[:1200])


locals {
  base_path = "${var.artifact_root}/${var.environment}"
}

resource "local_file" "storage_manifest" {
  filename = "${local.base_path}/storage_manifest.txt"
  content  = "bucket=${var.inventory_bucket_name}\npath=incoming/YYYY-MM-DD/inventory.csv\n"
}

resource "local_file" "mlflow_manifest" {
  filename = "${local.base_path}/mlflow_manifest.txt"
  content  = "tracking_uri=file:../mlruns\nartifact_store=../models\n"
}

resource "local_file" "airflow_manifest" {
  filename = "${local.base_path}/airflow_manifest.txt"
  content  = "dag_id=inventory_retrain_pipeline\nsensor=S3KeySensor_or_local_FileSensor\n"
}


--- terraform plan evidence ---
Terraform is not installed in this local environment.

Expected command:
terraform plan -out=tfplan
terraform show -no-color tfplan > ../reports/terraform_plan.txt

Demo planned resources:
- local_file.storage_manifest
- local_file.mlflow_manifest
- local_file.airflow_manifest

Plan intent:
- create local demo manifests for storage / registr

## 5. SLI/SLO и риски

| Уровень | SLI | Источник | Normal/SLO | Warning | Critical | Действие |
|---|---|---|---|---|---|---|
| Бизнес | доля SKU с прогнозом на завтра | output prediction batch | `>= 95%` SKU ежедневно | `90-95%` | `< 90%` | перезапуск DAG / ручной расчет top-SKU |
| Бизнес | доля дефицитных SKU, найденных системой | stockout labels | `>= 80%` за неделю | `65-80%` | `< 65%` | проверить продажи / пересмотреть модель |
| Модель/данные | MAPE прогноза остатков | validation report | `<= 15%` | `15-25%` | `> 25%` | не регистрировать модель / оставить baseline |
| Модель/данные | data validation pass rate | validation task | `100%` critical checks | один warning | любой critical fail | остановить train / incident по данным |
| Код/API | p95 latency прогноза | service logs / synthetic probe | `< 300 ms` за день | `300-1000 ms` | `> 1000 ms` | bottleneck analysis / rollback версии |
| Инфраструктура | successful DAG run rate | Airflow metadata | `>= 99%` daily runs за месяц | 1 падение | 2 падения подряд | разобрать логи / storage / registry |
| Инфраструктура | storage availability | S3/MinIO health | `>= 99.5%` за месяц | retry | нет доступа к batch | не запускать training / incident |

**Риски:**

| risk | уровень | как детектим | что делаем |
|---|---|---|---|
| новый batch не пришел | infra/data | sensor timeout | skip training + уведомление |
| batch с плохой схемой | data/code | validation fail | stop DAG до train |
| модель хуже baseline | model | compare task | `skip_deploy` |
| latency выросла | code/infra | MDD / monitoring | cache / optimize / rollback |
| Terraform удаляет лишнее | infra | review `plan` | approval до apply |


## 6. MDD: latency decision

Метрика: `latency_ms` для прогноза запасов.

Гипотезы:

- H0: latency не выросла статистически значимо
- H1: latency выросла статистически значимо
- alpha = 0.05
- test = Mann-Whitney U (без спора про нормальность)


In [4]:
from pathlib import Path

import pandas as pd
from scipy.stats import mannwhitneyu

ref = pd.read_csv("data/reference_latency.csv")["latency_ms"]
new = pd.read_csv("data/new_latency.csv")["latency_ms"]

stat, p_value = mannwhitneyu(new, ref, alternative="greater")
print({"stat": round(stat, 3), "p_value": round(p_value, 8)})
print("reference_median", round(ref.median(), 2))
print("new_median", round(new.median(), 2))
print(Path("reports/mdd_test_result.md").read_text(encoding="utf-8"))


{'stat': np.float64(2298.0), 'p_value': np.float64(0.0)}
reference_median 121.0
new_median 179.68
Generated at: 2026-05-24 17:22:41 MSK

# MDD latency test

- metric: `latency_ms`
- reference rows: `48`
- new rows: `48`
- reference median: `121.00`
- new median: `179.69`
- test: `Mann-Whitney U`, alternative=`greater`
- alpha: `0.05`
- statistic: `2298.000`
- p_value: `0.00000000`
- decision: `add cache before stock history read`

**Вывод:** p-value ниже 0.05, рост latency считаю статистически значимым.



![MDD latency distribution](reports/mdd_latency_distribution.png)

ADR: `adr/0001-latency-mdd-decision.md`

**Вывод:**

- p-value ниже 0.05
- latency выросла статистически значимо
- решение: добавить cache перед чтением истории остатков
- тяжелые lag features переносим в batch preprocessing


In [5]:
from pathlib import Path

print(Path("adr/0001-latency-mdd-decision.md").read_text(encoding="utf-8"))


Generated at: 2026-05-24 17:22:41 MSK

# ADR 0001: решение по росту latency прогноза запасов

## Status

Accepted

## Context

Есть два набора измерений latency:

- `data/reference_latency.csv` - нормальная история
- `data/new_latency.csv` - новый вариант расчета

Метрика системная: latency прогноза запасов. Это не accuracy модели, а скорость работы ML-системы.

## Decision

- H0: latency не выросла статистически значимо
- H1: latency выросла статистически значимо
- test: Mann-Whitney U, alternative=`greater`
- alpha: `0.05`
- p-value: `0.00000000`

Decision: добавить cache перед чтением истории остатков + перенести расчет тяжелых lag features в batch preprocessing.

## Consequences

Плюсы:

- меньше p95 latency на inference path
- меньше повторных чтений истории по одному SKU
- проще держать SLO `< 300 ms`

Минусы:

- появляется cache invalidation
- нужен контроль свежести batch features

Что мониторим дальше:

- p95 latency прогноза
- долю stale cache
- MAPE после переноса lag featur

## 7. Итоговый вывод

**Итого:**

- для склада выбрал batch + reactive retraining, т.к. остатки приходят пачками
- Airflow ждет новый batch и запускает ML-процесс
- CI/CD и Airflow разделены: CI проверяет код / DAG / Terraform, Airflow запускает long-running training
- новая модель проходит validation + compare с baseline
- если метрика хуже порога, остается старая модель
- Terraform нужен не для красоты, а чтобы storage / registry / deps были описаны кодом
- MDD сделал по latency: два распределения -> H0/H1 -> p-value -> ADR

**Checklist:**

- [x] архитектуры сравнены
- [x] Airflow DAG с S3/file sensor есть
- [x] Terraform plan/destroy evidence есть
- [x] SLI/SLO на 3 уровнях есть
- [x] MDD + ADR есть
